# Etapa 6 v2 — resumo da calibração e achados até o momento

Reproduz os dois gráficos gerados na sessão de 2026-07-16 a partir do lote completo (1000 imagens)
da etapa 6 v2 (âncora pela aorta), e reúne um resumo textual dos achados de todo o percurso da
etapa 6 no Capítulo 2 — desde a 1ª tentativa (âncora pelo LCC, descartada) até o resultado atual
(âncora pela aorta, calibrada, mas com cobertura <100% e `Rv` baixo).

Não reprocessa nada pesado — só lê os resultados já salvos em
`output/bounding_box/aorta_anchor/` (produzidos por
`uv run python src/run_aorta_anchor.py --splits config/imagecas_splits_train90.json`, ~10h de
execução) e reaproveita as funções de plot de
[plot_aorta_anchor_extremes.py](plot_aorta_anchor_extremes.py), para não duplicar o código dos
gráficos entre script e notebook.

Ver [docs/step6v2_aorta_anchor.md](../../docs/step6v2_aorta_anchor.md) para a documentação
completa (racional da âncora pela aorta, achado sobre o LCC, achado sobre GPU, achado sobre o
`Rv`).

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()
print(f"REPO_ROOT = {REPO_ROOT}")


In [ ]:
import json
import random

import numpy as np
import pandas as pd

from eda.plot_aorta_anchor_extremes import (
    DOCS_IMG_DIR,
    OUTPUT_DIR,
    SPLITS_STEM,
    plot_offset_histograms,
    plot_worst_coverage_cases,
)
from utils.utils.nifti_io import load_raw_img_and_label

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"{OUTPUT_DIR} não encontrado — rode primeiro "
        "`uv run python src/run_aorta_anchor.py --splits config/imagecas_splits_train90.json` "
        "(demora ~10h; os resultados já ficam salvos incrementalmente)."
    )


## 1. Carregar os resultados do lote (994/1000 casos, calibração com split `train90`)

In [ ]:
all_cases = pd.read_csv(OUTPUT_DIR / "all_cases.csv")
ok = all_cases[all_cases["status"] == "ok"].copy()
for axis in ("y", "x", "z"):
    ok[f"offset_min_{axis}"] = ok[f"label_{axis}_min"] - ok[f"anchor_{axis}"]
    ok[f"offset_max_{axis}"] = ok[f"label_{axis}_max"] - ok[f"anchor_{axis}"]

calibration = json.loads((OUTPUT_DIR / f"calibration_{SPLITS_STEM}.json").read_text(encoding="utf-8"))
splits = json.loads((REPO_ROOT / "config" / f"{SPLITS_STEM}.json").read_text(encoding="utf-8"))["splits"]
train_ids = {int(img_id) for img_id in splits["train"]}

val = pd.read_csv(OUTPUT_DIR / f"coverage_val_{SPLITS_STEM}.csv")
test = pd.read_csv(OUTPUT_DIR / f"coverage_test_{SPLITS_STEM}.csv")
val["split"] = "val"
test["split"] = "test"
coverage = pd.concat([val, test], ignore_index=True)

print(f"Casos processados com sucesso: {len(ok)} / {len(all_cases)}")
print(f"Casos de treino usados na calibração: {len(ok[ok['img_id'].isin(train_ids)])}")
print(f"Casos val+test (fora da calibração): {len(coverage)}")


## 2. Histograma dos offsets por eixo

Para cada eixo, mostra a distribuição de `label_min − âncora` e `label_max − âncora` entre todos
os 994 casos, com o limite calibrado (definido só pelos 88 casos de treino) marcado — e o caso
específico que definiu cada extremo. Reaproveita `plot_offset_histograms()` de
`plot_aorta_anchor_extremes.py` (mesma função usada para gerar
`docs/img/step6v2_offset_histograms.png`).

In [ ]:
plot_offset_histograms(ok, calibration)


## 3. Piores casos de cobertura (val + test, fora da calibração)

Os 20 casos que mais ultrapassam o bbox calibrado, e por qual eixo. Reaproveita
`plot_worst_coverage_cases()` (mesma função usada para gerar
`docs/img/step6v2_worst_coverage_cases.png`); retorna também o detalhe por caso em `excess_df`.

In [ ]:
excess_df = plot_worst_coverage_cases(coverage, calibration)

num_val_fail = int((excess_df.loc[excess_df["split"] == "val", "max_excess"] > 0).sum())
num_test_fail = int((excess_df.loc[excess_df["split"] == "test", "max_excess"] > 0).sum())
print(f"Falhas de cobertura — val: {num_val_fail}/{len(val)} ({num_val_fail/len(val):.2%})")
print(f"Falhas de cobertura — test: {num_test_fail}/{len(test)} ({num_test_fail/len(test):.2%})")


## 4. `Rv` (redução volumétrica) com o bbox calibrado

Estimativa amostral (60 casos reais, seed fixa para reprodutibilidade) do `Rv = 1 − V_bbox/V_original`,
usando o espaçamento físico de cada caso (a extensão do bbox calibrado é fixa em voxels, mas o
espaçamento físico por paciente varia, então o `Rv` em si varia por caso).

In [ ]:
extent_reduced = np.array([calibration["bbox_extent_reduced"][a] for a in ("y", "x", "z")])
downscale_factors = np.array(calibration["downscale_factors"])

random.seed(42)
sample_ids = random.sample(list(ok["img_id"]), 60)
dataset_root = REPO_ROOT / "dataset"
img_paths = {int(p.name.split(".")[0]): p for p in dataset_root.glob("*/*.img.nii.gz")}

rvs = []
for img_id in sample_ids:
    nii_img, _ = load_raw_img_and_label(str(img_paths[img_id]))
    spacing = np.array(nii_img.header.get_zooms())
    orig_shape = np.array(nii_img.shape)
    V_original = float(np.prod(orig_shape) * np.prod(spacing))
    scaled_spacing = spacing * downscale_factors
    V_bbox = float(np.prod(extent_reduced) * np.prod(scaled_spacing))
    rvs.append(1.0 - (V_bbox / V_original))

rvs = np.array(rvs)
print(f"Amostra: {len(rvs)} casos")
print(f"Rv médio:   {rvs.mean():.2%}")
print(f"Rv mediana: {np.median(rvs):.2%}")
print(f"Rv min-max: {rvs.min():.2%} a {rvs.max():.2%}")

bbox_extent_full = extent_reduced * downscale_factors
print()
print(f"Bbox calibrado em resolução integral (y,x,z): {tuple(bbox_extent_full.round(1))} voxels")
print("Imagens originais deste dataset costumam ter shape ~512x512x(200-300).")


## 5. Resumo dos achados até o momento (Capítulo 2 — pipeline de bounding box cardíaco)

**Etapas 1-4** (redução dimensional, threshold HU, maior componente conectado, empilhamento) —
implementadas e estáveis desde o início do capítulo. Produzem `lcc_image`/`lcc_mask`, usados como
entrada para todo o resto.

**Etapa 5** (centróide geométrico) — implementada e testada (`compute_geometric_centroid`).
Continua em uso, mas hoje aplicada à máscara da aorta, não mais ao `lcc_mask`.

**Etapa 6, 1ª tentativa — âncora pelo centróide do `lcc_mask`.** ❌ Achado bloqueante: o `lcc_mask`
isola o tronco inteiro (pele, parede torácica, mediastino), não o coração — não há fronteira de
densidade HU entre essas estruturas. 100% das fatias tocam a borda da imagem nos casos testados.
Substituída. Ver `docs/step6_envelope.md`.

**Etapa 6, 2ª tentativa — âncora pelo centróide da máscara da aorta** (Hough Circle Transform +
Level Set, código herdado do pipeline de segmentação arterial de Silva et al., reintroduzido só
para esse fim). O tamanho fixo do bbox é calibrado empiricamente: para cada eixo, o offset
mínimo/máximo observado entre os casos de treino, em relação à âncora. Determinístico (a âncora vem
de processamento de imagem, sem componente aprendido; os offsets são uma constante calculada uma
única vez via min/max sobre o treino).

- GPU local (RTX 4060) testada e **descartada**: acelera ~26%, mas troca o algoritmo de detecção de
  bordas (Canny → Sobel+percentil), mudando resultados, não só velocidade. Mantido `USE_GPU=false`.
- Lote completo (1000 imagens) rodado com o maior split de treino disponível (`train90`, 88 casos
  válidos) — decisão para reduzir o risco de calibrar com amostra pequena demais.
- **Resultado, e novo achado bloqueante**: cobertura em val/test **não chega a 100%** (~95-97%) e o
  `Rv` fica em torno de **5%** (às vezes negativo) — o bbox calibrado é quase do tamanho da imagem
  original. Causa: a agregação por mínimo/máximo é **independente por eixo**, então o bbox final
  precisa satisfazer o extremo de y de um paciente, o de x de outro, o de z de um terceiro — uma
  combinação que nenhum paciente real atinge sozinho, mas que o método produz mesmo assim. Mais
  casos de treino tende a piorar esse efeito, não corrigir.

**Isso ameaça a hipótese central do capítulo** (ganho simultâneo em qualidade *e* eficiência
computacional) — um `Rv` de ~5% não é redução de volume relevante.

**Decisão pendente**: como corrigir a calibração antes de implementar as etapas 7-9 (posicionamento,
corte, validação final com `Rv` oficial). Direções em discussão, não decididas — ver
`docs/step6v2_aorta_anchor.md`, seção "Possíveis direções":
1. Repensar a agregação (correlacionar eixos por paciente em vez de min/max independente).
2. Trocar a âncora para o ponto médio dos óstios coronarianos.
3. Investigar os casos mais extremos individualmente (outliers anatômicos vs. problema de label).
4. Reconsiderar se um bbox retangular de tamanho fixo é viável para este objetivo.